# Combinator Patterns in Nock

The Nock opcodes are simple rules which are concerned with manipulating and navigating binary trees, and with evaluating expressions based on these structures.  This means that they are fundamentally a sort of tree-based combinator language.  Combinators are functions that operate on other functions or data structures, allowing complex operations to be built up from simple, reusable components.  Well-known combinator systems include the [$SKI$ combinator calculus](https://en.wikipedia.org/wiki/SKI_combinator_calculus) and the [$BCKW$ combinator calculus](https://en.wikipedia.org/wiki/B,_C,_K,_W_system).

A complete combinator calculus lets you express any computation purely in terms of combinators, without the need for variables or other external constructs.  That is, they are [Turing complete](../specification/deep-nock.md).  Nock is also Turing complete, and has enough in common with the $SKI$ combinator calculus that from [its earliest public announcement](http://lambda-the-ultimate.org/node/1269), commenters noted the similarities and explored the connections between the two systems.  Mario B. remarked,

> My first impression is that U is a combinator calculus, except for * and > operators. The best known of the combinator calculi is SK combinator calculus, which you should have no problem googling for. It can be described very briefly, in your notation, as follows:

```
[name]   [pattern]            [definition]
(I)      (I $a)               $a
(K)      (K $a $b)            $b
(S)      (S $a $b $c)         ($a $c ($b $c))
```

Curtis Yarvin, who devised Nock, said at that time in response:

> I have not really worked with combinator models, but my general impression is that it takes essentially an infinite amount of syntactic sugar to turn them into a programming language.

The rough analogy between the $S$ combinator and Nock opcode 2 continued for two decades, but ~lagrev-nocfep recently formalized the connection by showing how the behavior of the $S$ combinator can be directly represented using Nock's primitive operations.

Taking $SKI$ to be their standard expressions:

```
S x y z = x z (y z) 
K x y   = x 
I x     = x
```

he showed that each of the $SKI$ combinators can be implemented using Nock's primitive operations.  Here we write that compiler using bracket abstraction:

```
[[S]]   = [1 S]   = [1 [1 [1 2]] [1 [0 1]] [[1 1] [0 1]]] 
[[K]]   = [1 K]   = [1 [1 1] [0 1]] 
[[I]]   = [1 I]   = [1 0 1] 
[[A B]] = [2 [[B]] [[A]]]
```

Opcode 2 does the applying.  `*[s 2 b a]` is `*[*[s b] *[s a]]`, so `[2 [[B]] [[A]]]` takes the value of `A`, which is a formula, and runs it with the value of `B` as its subject.  Applying a combinator to an argument means running its formula on the argument.

Each combinator's value is built to take its arguments one at a time.  `K` run on *x* gives `[1 x]`, a formula that ignores its next argument and returns *x*.  `I` is `[0 1]`, which returns its argument.  `S` takes three steps:

```{nockmark}
[1 {{[1 [1 2]]|op}}
%% note op run on *x*, gives `[1 2]`, which on *y* gives 2, the opcode
   {{[1 [0 1]]|y}}
%% note y gives `[0 1]`, which on *y* gives *y* itself
   {{[[1 1] [0 1]]|x}}]
%% note x gives `[1 x]`, which on *y* gives *x* back
```

Run on *x*, the three pieces autocons into a new formula; run on *y*, that one makes an opcode 2 formula:

```
S x     = *[x S]                 = [[1 2] [0 1] 1 x]
S x y   = *[y [1 2] [0 1] 1 x]   = [2 y x]
S x y z = *[z 2 y x]             = *[*[z y] *[z x]]
```

The last line runs `*[z x]` (that is, *x z*) as a formula on `*[z y]` (*y z*):  it is `x z (y z)`.

For example, `S K K` acts as `I`.  Compiled, it is

```{nockmark}
[2 {{[1 <K>]|k2}}
%% note k2 `[[K]]`, the second argument
   [2 {{[1 <K>]|k1}}
%% note k1 `[[K]]`, the first argument
      {{[1 <S>]|s}}]]
%% note s `[[S]]`: applied to *K* gives *S K*, then to *K* gives `[2 K K]`
%% skip
<K> = [[1 1] [0 1]]
<S> = [[1 [1 2]] [1 [0 1]] [[1 1] [0 1]]]
```

Its value is `[2 K K]`, printed `[2 [[1 1] 0 1] [1 1] 0 1]`.  Run on any *z*, it gives `*[*[z K] *[z K]]` = `*[[1 z] [1 z]]` = *z*, just as `I` would:  run on `42` it gives `42`.  Applying it to `I` as well, `[2 [1 0 1] [[S K K]]]`, gives `[0 1]`, the value of `I`.

* [~lagrev-nocfep (2026) “Representing the SKI Combinator Calculus in the Nock Instruction Set Architecture”, *Urbit Systems Technical Journal 3*: 2.](https://urbitsystems.tech/article/v03-i02/representing-the-ski-combinator-calculus-in-the-nock-instruction-set-architecture)